In [1]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account
from time import time

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [3]:
client = bigquery.Client()

def show_amount_of_data_scanned(query):
    # dry_run lets us see how much data the query uses without running it
    dry_run_config = bigquery.QueryJobConfig(dry_run=True)
    query_job = client.query(query, job_config=dry_run_config)
    print('Data processed: {} GB'.format(round(query_job.total_bytes_processed / 10**9, 3)))
    
def show_time_to_run(query):
    time_config = bigquery.QueryJobConfig(use_query_cache=False)
    start = time()
    query_result = client.query(query, job_config=time_config).result()
    end = time()
    print('Time to run: {} seconds'.format(round(end-start, 3)))

In [4]:
# Strategy 1: Only select the columns you need
#
# SELECT * tells BigQuery to read every column in the table.
# The contents table includes heavy columns like the full file source code,
# which means BigQuery has to scan all of that data — even if you don't use it.
star_query = "SELECT * FROM `bigquery-public-data.github_repos.contents`"
show_amount_of_data_scanned(star_query)
# Result: ~2682 GB scanned — reads every column including large source code blobs

# Selecting only the two columns we actually need (size and binary) tells BigQuery
# to skip all the other columns entirely. BigQuery is columnar storage, so it
# physically reads only the columns you name — nothing else.
basic_query = "SELECT size, binary FROM `bigquery-public-data.github_repos.contents`"
show_amount_of_data_scanned(basic_query)
# Result: ~2.5 GB scanned — over 1000x less data for the same number of rows
# In this case, we see a 1000X reduction in data being scanned to complete the query, 
# because the raw data contained a text field that was 1000X larger than the fields we might need.

Data processed: 2682.118 GB
Data processed: 2.531 GB


In [5]:
# Strategy 2: Avoid columns you don't need by using smarter filtering
#
# Both queries want the same result: trips where start and end stations differ,
# with average duration per route. The difference is WHICH columns they use
# to express that logic.

# more_data_query filters and groups by station ID columns (integers),
# then uses MIN() to look up the station NAME for display.
# This forces BigQuery to scan FOUR columns: start_station_id, end_station_id,
# start_station_name, end_station_name — plus duration_sec.
more_data_query = """
                  SELECT MIN(start_station_name) AS start_station_name,
                      MIN(end_station_name) AS end_station_name,
                      AVG(duration_sec) AS avg_duration_sec
                  FROM `bigquery-public-data.san_francisco.bikeshare_trips`
                  WHERE start_station_id != end_station_id 
                  GROUP BY start_station_id, end_station_id
                  LIMIT 10  
                  """
show_amount_of_data_scanned(more_data_query)
# Result: ~0.076 GB

# less_data_query filters and groups directly by station NAME.
# Since station name and station ID have a 1:1 relationship (same information,
# different columns), the result is identical — but now BigQuery only needs to
# scan THREE columns: start_station_name, end_station_name, duration_sec.
# The ID columns are never touched at all.
less_data_query = """
                  SELECT start_station_name,
                      end_station_name,
                      AVG(duration_sec) AS avg_duration_sec                  
                  FROM `bigquery-public-data.san_francisco.bikeshare_trips`
                  WHERE start_station_name != end_station_name
                  GROUP BY start_station_name, end_station_name
                  LIMIT 10
                  """
show_amount_of_data_scanned(less_data_query)
# Result: ~0.06 GB — smaller because we eliminated the two ID columns entirely

Data processed: 0.076 GB
Data processed: 0.06 GB


In [6]:
# Strategy 3: Avoid N:N JOINs — filter and aggregate BEFORE joining
#
# The commits table has many rows per repo (one per commit).
# The files table also has many rows per repo (one per file).
# Joining them directly on repo_name creates an N:N JOIN:
# every commit row matches every file row in the same repo,
# exploding the intermediate result into a massive table before aggregating.
#
# big_join_query does this the expensive way:
# it joins the full commits and files tables first (N:N explosion),
# then counts distinct committers and files from the giant joined result.
big_join_query = """
                 SELECT repo,
                     COUNT(DISTINCT c.committer.name) as num_committers,
                     COUNT(DISTINCT f.id) AS num_files
                 FROM `bigquery-public-data.github_repos.commits` AS c,
                     UNNEST(c.repo_name) AS repo
                 INNER JOIN `bigquery-public-data.github_repos.files` AS f
                     ON f.repo_name = repo
                 WHERE f.repo_name IN ( 'tensorflow/tensorflow', 'facebook/react', 'twbs/bootstrap', 'apple/swift', 'Microsoft/vscode', 'torvalds/linux')
                 GROUP BY repo
                 ORDER BY repo
                 """
show_time_to_run(big_join_query)
# Result: ~14 seconds — slow because the JOIN runs on full-size tables

# small_join_query does this the efficient way using CTEs (WITH clauses):
# Step 1 — aggregate commits down to one row per repo BEFORE the join
# Step 2 — aggregate files down to one row per repo BEFORE the join
# Step 3 — join the two pre-aggregated tables, which now have only 6 rows each (1:1 JOIN)
#
# By shrinking each table to its summary FIRST, the JOIN never sees the raw N:N explosion.
small_join_query = """
                   WITH commits AS
                   (
                   SELECT COUNT(DISTINCT committer.name) AS num_committers, repo
                   FROM `bigquery-public-data.github_repos.commits`,
                       UNNEST(repo_name) as repo
                   WHERE repo IN ( 'tensorflow/tensorflow', 'facebook/react', 'twbs/bootstrap', 'apple/swift', 'Microsoft/vscode', 'torvalds/linux')
                   GROUP BY repo
                   ),
                   files AS 
                   (
                   SELECT COUNT(DISTINCT id) AS num_files, repo_name as repo
                   FROM `bigquery-public-data.github_repos.files`
                   WHERE repo_name IN ( 'tensorflow/tensorflow', 'facebook/react', 'twbs/bootstrap', 'apple/swift', 'Microsoft/vscode', 'torvalds/linux')
                   GROUP BY repo
                   )
                   SELECT commits.repo, commits.num_committers, files.num_files
                   FROM commits 
                   INNER JOIN files
                       ON commits.repo = files.repo
                   ORDER BY repo
                   """

show_time_to_run(small_join_query)
# Result: ~3 seconds — 4x faster because the JOIN is now 1:1 (6 rows x 6 rows)

Time to run: 16.131 seconds
Time to run: 3.452 seconds
